# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: This notebook contains code outlining our data collection approach. 
</div>

In [ ]:
import requests
import json
import os
import pandas as pd
from pathlib import Path
import shelve
import math
from collections import defaultdict

## Data Collection Methodology

1. **Using Google Places API, find nightclubs for a certain area**

2. **Collect up to twenty stations within a radius of 1 kilometre from the nightclub:**
    - allows for up to a twelve minute walk at a typical walking pace of 3mph.

3. **Filter stations according to direction**
    - Preserving the labels from step 2, divide the area surrounding a club into eight 45 degree sections. For each section, take _the two closest stations_ to the nightclub **but always include** train stations (typically larger transport hubs).

4. **Generate Routes:**
    - Use Google Routes API to calculate routes for stations (computeroutematrix allows for batch routing, saving costs).
    Label routes accordingly:
        - 'very short' ( under 5 mins).
        - 'short' (between 5-8 mins).
        - 'long' (between 8-15 mins).

## 1. Gather nightclub data for a popular area in London

This [TimeOut list ](https://www.timeout.com/london/clubs/the-best-clubs-in-london) features 5 clubs in Dalston, the most for any area on the list. This test will use Dalston to generate clubs.

In [14]:
# Constants
places_url = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')

In [3]:
# Request google places text search for nightclubs in Dalston

headers = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': '*',
    # IMPORTANT: using '*' for FieldMask returns all fields. I am only using this for the test - don't use when we know what fields we want returned because it could increase costs.
}

data = {
    'textQuery': 'nightclubs in Dalston, London, England'
}

response = requests.post(url=places_url, headers=headers, data=json.dumps(data))

if response.status_code == 200:
    print('Success!')
else:
    print(f'Error: {response.status_code}: {response.text}')

Success!


In [ ]:
with open('data/raw/dalston_nightclubs.json', w) as f:
    json.dump(response.json(), f, indent=2)

## 2. Collect up to twenty stations within a radius of 1 kilometre from the nightclub

In [17]:
with open('data/raw/dalston_nightclubs.json', 'r') as f:
    data = json.load(f)

In [20]:
first_club_lat, first_club_long = (
    data['places'][0]['location']['latitude'],
    data['places'][0]['location']['longitude']
)


In [21]:
# Test constants

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places,places.displayName,places.location,places.primaryType',
    }
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [22]:
# Test request

data = {
        'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
        'maxResultCount' : 20,
        'rankPreference': 'DISTANCE',
        'locationRestriction' : {
            'circle': {
                'center' : {
                    'latitude' : first_club_lat,
                    'longitude' : first_club_long
                },
                'radius' : 1000
            }
        }
    }
        

response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data)

if response.status_code != 200:
    print(f'Error: {response.status_code}: {response.text}')

In [23]:
with open('data/raw/stations_from_dalston_superstore', 'w') as f:
    json.dump(response.json(), f, indent=2)

## 3. Filter stations according to direction

In [2]:
with open('data/raw/stations_from_dalston_superstore', 'r') as f:
    stations = json.load(f)

In [20]:
stations.keys()

dict_keys(['places'])

In [ ]:
first_club_lat, first_club_long = (
    stations['places'][0]['location']['latitude'],
    stations['places'][0]['location']['longitude']
)

In [23]:
# define a function that returns a bearing fro point 1 to point 2 in degrees (0-360) where 0 = North and 90 = East

def bearing_degrees(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_lon = math.radians(lon2 - lon1)

    x = math.sin(delta_lon) * math.cos(phi2)
    y = (
        math.cos(phi1) * math.sin(phi2)
        - math.sin(phi1) * math.cos(phi2) * math.cos(delta_lon)
    )

    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360

In [24]:
# define a function that assigns each station to one of 8 sectors

def sector_index(bearing):
    return int(bearing // 45)

In [ ]:
# define haversine to calculate the shortest distance between two points on a sphere

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [66]:
def directional_filtering(stations, club_lat, club_lon, max_per_sector=2):
    sectors = defaultdict(list)
    train_stations = []

    for station in stations['places']:
        lat = station['location']['latitude']
        lon = station['location']['longitude']

        # 1. compute straight-line distance in metres from club to station

        station['distance_m'] = haversine(club_lat, club_lon, lat, lon)

        # 2. assign station to a bearing sector relative to the club

        bearing = bearing_degrees(club_lat, club_lon, lat, lon)
        sector = sector_index(bearing)

        station['bearing'] = bearing
        station['sector'] = sector
        
        # 3. split by station type

        if station['primaryType'] in ('train_station', 'subway_station'):
            train_stations.append(station)
        else:
            sectors[sector].append(station)

    selected = []

    # select closest bus stations per sector

    for sector_stations in sectors.values():
        sector_stations.sort(key=lambda s: s['distance_m'])
        selected.extend(sector_stations[:max_per_sector])

    # always include train/subway stations

    selected_ids = {
        (
            s['location']['latitude'], s['location']['longitude']
            ) 
            for s in selected
        }

    for station in train_stations:
        key = (station['location']['latitude'], station['location']['longitude'])
        if key not in selected_ids:
            selected.append(station)

    return selected

In [ ]:
filtered_dss_stations = directional_filtering(
    stations=stations,
    club_lat=first_club_lat,
    club_lon=first_club_long,
    max_per_sector=2
)

In [ ]:
for station in filtered_dss_stations: 
    print(f'This station is in sector: {station['sector']}')

In [53]:
first_station_lat, first_station_long = (
    filtered_dss_stations[9]['location']['latitude'], 
    filtered_dss_stations[9]['location']['longitude']
)

## 4. Generate Routes for Stations

In [55]:
# Test constants

API_KEY = os.getenv('API_KEY')
ROUTES_URL = 'https://routes.googleapis.com/directions/v2:computeRoutes'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'routes.duration,routes.distanceMeters,routes.polyline',
    }

In [56]:
data = {
    'origin':{
        'location':{
            'latLng':{
                'latitude': first_club_lat,
                'longitude': first_club_long
            }
        }
    },

    'destination':{
        'location':{
            'latLng':{
                'latitude': first_station_lat,
                'longitude': first_station_long
            }
        }
    },

    'travelMode': 'WALK',
    'polylineEncoding': 'GEO_JSON_LINESTRING',
    'polylineQuality': 'OVERVIEW'
}

response = requests.post(url=ROUTES_URL, headers=HEADERS, json=data)

In [59]:
with open('data/raw/test_routes_data.json', 'w') as f:
    json.dump(response.json(), f, indent=2)

## 5. Join Routes Data to Police API

In [4]:
with open('data/raw/test_routes_data.json', 'r') as f:
    routes = json.load(f)

In [ ]:
routes['routes'][0]['polyline']['geoJsonLinestring']['coordinates'][0][1]

-0.0750923

In [80]:
def reduce_polyline(route, min_spacing_m):
# This function slims down the polyline provided by google maps api to reduce number of route vertices for Police API
    polyline = route['routes'][0]['polyline']['geoJsonLinestring']['coordinates']
    sampled = [polyline[0]]
    acc_dist = 0.0
    prev_lon, prev_lat = polyline[0]

    for curr_lon, curr_lat in polyline[1:]:
        step_distance = haversine(prev_lat, prev_lon, curr_lat, curr_lon)
        acc_dist += step_distance

        if acc_dist >= min_spacing_m:
            sampled.append([curr_lon, curr_lat])
            acc_dist = 0.0

        prev_lon, prev_lat = curr_lon, curr_lat

    if sampled[-1] != polyline[-1]:
        sampled.append(polyline[-1])

    return sampled

In [83]:
test_reduce = reduce_polyline(routes, 30)
print(len(test_reduce))
print(len(routes['routes'][0]['polyline']['geoJsonLinestring']['coordinates']))

13
21


In [ ]:
# Police API request

POLICE_URL = 'https://data.police.uk/api/crimes-street/all-crime?date=2024-01'
poly=

## Function Design

In [32]:
# Constants for google places nearby API

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places.displayName,places.location,places.primaryType',
    }
CACHE_DIR = Path('/files/group_project/cache_dir')
CACHE_FILE = CACHE_DIR / 'station_cache_v1.json'
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [11]:
# check if cache file exists / load cache file

if CACHE_FILE.exists():
    station_cache = json.loads(CACHE_FILE.read_text())
else:
    station_cache = {}

In [33]:
# Define a function to request google places nearby API for nearest public transport stations for a location

def get_nearest_station(lat, lon):

    cache_key = f'{round(lat, ROUNDING)},{round(lon, ROUNDING)}'

    if cache_key in station_cache:
        return station_cache[cache_key]
    # avoid API calls if location already used

    data = {
        'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
        'maxResultCount' : 1,
        'rankPreference': 'DISTANCE',
        'locationBias' : {
            'point': {
                'latitude' : lat,
                'longitude' : lon
            }
        }
    }
        

    response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data)

    if response.status_code != 200:
        print(f'Error: {response.status_code}: {response.text}')
        return None

    places = response.json().get('places', [])
        
    if places:
        station = {
            'name': places[0]['displayName']['text'],
            'latitude': places[0]['location']['latitude'],
            'longitude': places[0]['location']['longitude']
        }  
            
        station_cache[cache_key] = station
        print(json.dumps(response.json(), indent=2))
        return station

    # no station found at any radius
    station_cache[cache_key] = None
    return None

In [34]:
# test  nearest station API collection with first dalston clubs

test_station = get_nearest_station(first_club_lat, first_club_long)
print(test_station)

None


### Limitations and Reflections before NB02

_Data Quality of Police API_

The [Police API](https://data.police.uk/about/) lists some known issues regarding data quality. A particularly important issue for our purposes is that there are inconsistent geocoding policies in police forces. Estimates of geocoding accuracy in different forces range from 60% to 97%. We cannot filter for confident location data so we will acknowledge this limitation in our report.

_Google Route Matrix API Version_

The 'Walk' option of the [RouteTravelMode](https://developers.google.com/maps/documentation/routes/reference/rest/v2/RouteTravelMode?_gl=1*1v9v6vk*_up*MQ..*_ga*MTQ3MDM2MzQyLjE3NjcyMzk3MzQ.*_ga_NRWSTWS78N*czE3NjcyMzk3MzMkbzEkZzEkdDE3NjcyNDAyMTkkajYwJGwwJGgw*_ga_SM8HXJ53K2*czE3NjcyMzk3MzMkbzEkZzAkdDE3NjcyNDAyMTckajYwJGwwJGgw) is in beta. Google Maps state that it might sometimes be missing clear sidewalk and pedestrian paths. 